# 🏢 Entity 360 Dashboard

**Reconciliation Management System - dbt Model Showcase**

This notebook visualizes the Entity 360 dbt model built on the COCO_LIVE_DB reconciliation data. It demonstrates:
- Entity health distribution
- Variance analysis trends
- Reconciliation completion rates
- Risk assessment metrics

In [ ]:
%%sql -r health_summary
SELECT 
    entity_health_status,
    COUNT(*) as entity_count,
    SUM(total_assignments) as total_assignments,
    ROUND(AVG(reconciliation_completion_pct), 2) as avg_completion_pct,
    SUM(critical_variance_count) as critical_variances,
    SUM(high_severity_count) as high_severity_variances
FROM COCO_LIVE_DB.PUBLIC.MART_ENTITY_360
GROUP BY entity_health_status
ORDER BY entity_count DESC

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd

In [ ]:
fig = px.pie(
    health_summary, 
    values='ENTITY_COUNT', 
    names='ENTITY_HEALTH_STATUS',
    title='Entity Health Distribution',
    color='ENTITY_HEALTH_STATUS',
    color_discrete_map={
        'Critical': '#dc3545',
        'At Risk': '#ffc107', 
        'Needs Attention': '#17a2b8',
        'Healthy': '#28a745',
        'No Activity': '#6c757d'
    },
    hole=0.4
)
fig.update_traces(textposition='inside', textinfo='percent+label')
fig.update_layout(height=500)
fig.show()

In [ ]:
%%sql -r variance_by_health
SELECT 
    entity_health_status,
    SUM(total_variance_amount) as total_variance,
    SUM(critical_variance_count) as critical_count,
    SUM(high_severity_count) as high_count,
    ROUND(AVG(avg_variance_amount), 2) as avg_variance
FROM COCO_LIVE_DB.PUBLIC.MART_ENTITY_360
WHERE total_variance_amount > 0
GROUP BY entity_health_status
ORDER BY total_variance DESC

In [ ]:
fig = px.bar(
    variance_by_health,
    x='ENTITY_HEALTH_STATUS',
    y=['CRITICAL_COUNT', 'HIGH_COUNT'],
    title='Variance Severity by Entity Health Status',
    barmode='stack',
    color_discrete_map={'CRITICAL_COUNT': '#dc3545', 'HIGH_COUNT': '#ffc107'},
    labels={'value': 'Count', 'variable': 'Severity', 'ENTITY_HEALTH_STATUS': 'Health Status'}
)
fig.update_layout(height=450, legend_title='Severity Level')
fig.show()

In [ ]:
%%sql -r variance_trend
SELECT 
    period_year,
    period_quarter,
    COUNT(DISTINCT entity_id) as entities_affected,
    SUM(total_variance) as total_variance,
    SUM(critical_count) as critical_variances,
    SUM(severe_variance_count) as severe_variances,
    period_risk_level
FROM COCO_LIVE_DB.PUBLIC.MART_VARIANCE_SUMMARY
GROUP BY period_year, period_quarter, period_risk_level
ORDER BY period_year, period_quarter

In [ ]:
variance_trend['PERIOD'] = variance_trend['PERIOD_YEAR'].astype(str) + '-Q' + variance_trend['PERIOD_QUARTER'].astype(str)

fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=('Total Variance by Period', 'Entities Affected by Period'),
    vertical_spacing=0.15
)

fig.add_trace(
    go.Bar(
        x=variance_trend['PERIOD'], 
        y=variance_trend['TOTAL_VARIANCE'],
        name='Total Variance',
        marker_color='#3498db'
    ),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(
        x=variance_trend['PERIOD'], 
        y=variance_trend['ENTITIES_AFFECTED'],
        name='Entities Affected',
        mode='lines+markers',
        line=dict(color='#e74c3c', width=3),
        marker=dict(size=10)
    ),
    row=2, col=1
)

fig.update_layout(height=700, title_text='Variance Trends Over Time', showlegend=True)
fig.show()

In [ ]:
%%sql -r at_risk_entities
SELECT 
    entity_name,
    entity_code,
    entity_health_status,
    total_assignments,
    reconciled_count,
    reconciliation_completion_pct,
    total_variance_amount,
    critical_variance_count
FROM COCO_LIVE_DB.PUBLIC.MART_ENTITY_360
WHERE entity_health_status IN ('Critical', 'At Risk')
ORDER BY critical_variance_count DESC, total_variance_amount DESC
LIMIT 15

In [ ]:
fig = px.scatter(
    at_risk_entities,
    x='RECONCILIATION_COMPLETION_PCT',
    y='TOTAL_VARIANCE_AMOUNT',
    size='CRITICAL_VARIANCE_COUNT',
    color='ENTITY_HEALTH_STATUS',
    hover_name='ENTITY_NAME',
    title='At-Risk Entities: Completion Rate vs Variance',
    labels={
        'RECONCILIATION_COMPLETION_PCT': 'Completion Rate (%)',
        'TOTAL_VARIANCE_AMOUNT': 'Total Variance Amount',
        'CRITICAL_VARIANCE_COUNT': 'Critical Variances'
    },
    color_discrete_map={'Critical': '#dc3545', 'At Risk': '#ffc107'},
    size_max=50
)
fig.update_layout(height=500)
fig.show()

In [ ]:
%%sql -r completion_distribution
SELECT 
    CASE 
        WHEN reconciliation_completion_pct >= 90 THEN '90-100%'
        WHEN reconciliation_completion_pct >= 70 THEN '70-89%'
        WHEN reconciliation_completion_pct >= 50 THEN '50-69%'
        WHEN reconciliation_completion_pct >= 25 THEN '25-49%'
        ELSE '0-24%'
    END as completion_bucket,
    COUNT(*) as entity_count,
    SUM(total_assignments) as total_assignments
FROM COCO_LIVE_DB.PUBLIC.MART_ENTITY_360
WHERE total_assignments > 0
GROUP BY completion_bucket
ORDER BY completion_bucket DESC

In [ ]:
bucket_order = ['90-100%', '70-89%', '50-69%', '25-49%', '0-24%']
completion_distribution['COMPLETION_BUCKET'] = pd.Categorical(
    completion_distribution['COMPLETION_BUCKET'], 
    categories=bucket_order, 
    ordered=True
)
completion_distribution = completion_distribution.sort_values('COMPLETION_BUCKET')

colors = ['#28a745', '#7cb342', '#ffc107', '#ff9800', '#dc3545']

fig = px.bar(
    completion_distribution,
    x='COMPLETION_BUCKET',
    y='ENTITY_COUNT',
    color='COMPLETION_BUCKET',
    title='Reconciliation Completion Rate Distribution',
    labels={'ENTITY_COUNT': 'Number of Entities', 'COMPLETION_BUCKET': 'Completion Rate'},
    color_discrete_sequence=colors
)
fig.update_layout(height=450, showlegend=False)
fig.show()

In [ ]:
%%sql -r kpi_summary
SELECT 
    COUNT(*) as total_entities,
    SUM(total_assignments) as total_assignments,
    SUM(reconciled_count) as total_reconciled,
    ROUND(100.0 * SUM(reconciled_count) / NULLIF(SUM(total_assignments), 0), 2) as overall_completion_pct,
    SUM(critical_variance_count) as total_critical_variances,
    SUM(high_severity_count) as total_high_variances,
    ROUND(SUM(total_variance_amount), 2) as total_variance_amount
FROM COCO_LIVE_DB.PUBLIC.MART_ENTITY_360

In [ ]:
kpi = kpi_summary.iloc[0]

fig = go.Figure()

fig.add_trace(go.Indicator(
    mode="number",
    value=kpi['TOTAL_ENTITIES'],
    title={"text": "Total Entities"},
    domain={'x': [0, 0.25], 'y': [0.5, 1]}
))

fig.add_trace(go.Indicator(
    mode="number+delta",
    value=kpi['OVERALL_COMPLETION_PCT'],
    title={"text": "Overall Completion %"},
    delta={'reference': 80, 'relative': False, 'position': "bottom"},
    domain={'x': [0.25, 0.5], 'y': [0.5, 1]}
))

fig.add_trace(go.Indicator(
    mode="number",
    value=kpi['TOTAL_CRITICAL_VARIANCES'],
    title={"text": "Critical Variances"},
    number={'font': {'color': '#dc3545'}},
    domain={'x': [0.5, 0.75], 'y': [0.5, 1]}
))

fig.add_trace(go.Indicator(
    mode="number",
    value=kpi['TOTAL_VARIANCE_AMOUNT'],
    title={"text": "Total Variance ($)"},
    number={'prefix': "$", 'valueformat': ",.0f"},
    domain={'x': [0.75, 1], 'y': [0.5, 1]}
))

fig.update_layout(
    title='Key Performance Indicators',
    height=300,
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)'
)
fig.show()

---
## 📊 Summary

This dashboard showcases the **Entity 360 dbt model** built from the COCO_LIVE_DB reconciliation management system.

### Key Insights:
- **Health Distribution**: Visual breakdown of entities by health status
- **Variance Analysis**: Trends over time and severity breakdowns
- **Risk Assessment**: Scatter plot identifying high-risk entities
- **Completion Tracking**: Distribution of reconciliation completion rates
- **KPIs**: Real-time metrics for overall system health

### dbt Models Used:
- `mart_entity_360` - Comprehensive entity view with health scoring
- `mart_variance_summary` - Period-by-period variance analysis
- `semantic_entity_360` - Semantic view for Cortex Analyst integration